In [6]:
pip install pyyaml

  Using cached pyyaml-6.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.4 kB)
Using cached pyyaml-6.0.3-cp314-cp314-macosx_11_0_arm64.whl (173 kB)

[notice] A new release of pip is available: 26.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
# loader.py
# Reads a YAML system description and produces an SMCDEL .txt file.
# Requires: pip install pyyaml
#
# Usage (CLI):
#   python loader.py system_description.yaml --scenario 8
#   python loader.py system_description.yaml --scenario 8 --output out.smcdel.txt
#
# Usage (Python):
#   from loader import run_scenario
#   run_scenario("system_description.yaml", scenario_id=8)

import yaml
import sys
import argparse
from example.classes import Node, Relationship, Agent, EpistemicModel, SMCDELGenerator


# ==============================================================================
# LOADING
# ==============================================================================

def load_yaml(path: str) -> dict:
    with open(path, "r") as f:
        return yaml.safe_load(f)


def build_model(data: dict, tools_in_hand: list) -> tuple[EpistemicModel, dict]:
    """
    Build an EpistemicModel from the YAML data, selecting the agent whose
    tools match tools_in_hand.

    Returns:
        model       - the populated EpistemicModel
        node_lookup - dict mapping node name -> Node (for use in queries)
    """
    model = EpistemicModel()

    # 1. Nodes
    node_lookup: dict[str, Node] = {}
    for n in data.get("nodes", []):
        node = model.add_node(Node(name=n["name"], kind=n["kind"]))
        node_lookup[n["name"]] = node

    # 2. Relationships
    for r in data.get("relationships", []):
        child_node = node_lookup[r["child"]]
        parent_nodes = [node_lookup[p] for p in r.get("parents", [])]
        rel = Relationship(
            child=child_node,
            parents=parent_nodes,
            type=r["type"],
            formula=r.get("formula", None),
        )
        model.add_relationship(rel)

    # 3. Select the agent whose tool set matches tools_in_hand
    tools_set = set(tools_in_hand)
    selected_agent_data = None
    for a in data.get("agents", []):
        if set(a.get("tools", [])) == tools_set:
            selected_agent_data = a
            break

    if selected_agent_data is None:
        # Fall back to the agent with the fewest tools (no-tool technician)
        selected_agent_data = min(
            data.get("agents", []),
            key=lambda a: len(a.get("tools", []))
        )
        print(f"  [Warning] No agent found for tools {tools_in_hand}. "
              f"Falling back to '{selected_agent_data['name']}'.")

    agent = Agent(name=selected_agent_data["name"])
    for obs_name in selected_agent_data.get("observes", []):
        agent.observe(node_lookup[obs_name])
    model.add_agent(agent)

    return model, node_lookup


# ==============================================================================
# SCENARIO LOOKUP
# ==============================================================================

def get_scenario(data: dict, scenario_id: int) -> dict:
    """Find a scenario by ID. Raises ValueError if not found."""
    for s in data.get("scenarios", []):
        if s["id"] == scenario_id:
            return s
    available = [s["id"] for s in data.get("scenarios", [])]
    raise ValueError(
        f"Scenario ID {scenario_id} not found. Available IDs: {available}"
    )


# ==============================================================================
# QUERY BUILDING
# ==============================================================================

def build_queries(model, scenario, generator):
    observations = scenario.get("observations", {})
    agent = model.agents[0]

    ann_parts = []
    for node_name, is_faulty in observations.items():
        node = next((n for n in model.nodes if n.name == node_name), None)
        expr = generator._expr_cache.get(node.name)

        if node.kind == 'fault':
            ann = f"[! {expr}]" if is_faulty else f"[! ~{expr}]"
        else:
            # derived node — use expanded fault-ID expression
            ann = f"[! ({expr})]" if is_faulty else f"[! ~({expr})]"
        ann_parts.append(ann)

    ann_prefix = " ".join(ann_parts)

    obs_nodes   = [n for n in model.nodes if n.kind in ("observable", "measurement")]
    fault_nodes = [n for n in model.nodes if n.kind == "fault"]

    lines = ["\n-- ============================================================"]
    lines.append(f"-- Scenario {scenario['id']}: {scenario.get('description', '')}")
    lines.append(f"-- Ground truth faults: {scenario.get('fault_fns', [])}")
    lines.append(f"-- Agent: {agent.name}")
    lines.append("-- ============================================================")

    # Q1: surviving worlds
    lines.append("\n-- Worlds remaining after observations:")
    lines.append(f"WHERE?\n  {ann_prefix} Top")

    # Q2: does the agent know each observed node's value?
    for node_name, is_faulty in observations.items():
        node = next((n for n in model.nodes if n.name == node_name), None)
        if node is None:
            continue
        knows_str = f"({agent.name} knows that {node.id})" if is_faulty \
                    else f"({agent.name} knows that ~{node.id})"
        lines.append(f"\n-- Does {agent.name} know that {node.name} = {is_faulty}?")
        lines.append(f"VALID?\n  {ann_prefix} {knows_str}")

    # Q3: can the agent diagnose each fault?
    for node in fault_nodes:
        lines.append(f"\n-- Can {agent.name} determine whether {node.name} is faulty?")
        lines.append(f"VALID?\n  {ann_prefix} ({agent.name} knows whether {node.id})")

    return "\n".join(lines)


# ==============================================================================
# MAIN ENTRY POINT
# ==============================================================================

def run_scenario(
    yaml_path: str,
    scenario_id: int,
    output_path: str = None,
    print_output: bool = True
) -> str:
    """
    Load the YAML, select scenario by ID, build the model, generate SMCDEL.

    Args:
        yaml_path    - path to system_description.yaml
        scenario_id  - integer scenario ID to run
        output_path  - optional file path to write the result
        print_output - whether to print the result to stdout

    Returns:
        The full SMCDEL text as a string.
    """
    data = load_yaml(yaml_path)
    scenario = get_scenario(data, scenario_id)

    print(f"\nRunning scenario {scenario_id}: {scenario.get('description', '')}")
    print(f"  Tools in hand : {scenario.get('tools_in_hand', [])}")
    print(f"  Ground truth  : {scenario.get('fault_fns', [])}")

    model, node_lookup = build_model(data, scenario.get("tools_in_hand", []))

    generator = SMCDELGenerator(model)
    smcdel_body = generator.generate()

    queries = build_queries(model, scenario, generator)
    full_output = smcdel_body + "\n" + queries

    if output_path:
        with open(output_path, "w") as f:
            f.write(full_output)
        print(f"\n  SMCDEL file written to: {output_path}")

    if print_output:
        print("\n--- SMCDEL Output ---\n")
        print(full_output)

    return full_output


# ==============================================================================
# CLI
# ==============================================================================




In [4]:

run_scenario("example/circuit.yaml", scenario_id=7)


Running scenario 7: Battery depleted — PSU LED off, all module LEDs off, lamp off
  Tools in hand : ['multimeter']
  Ground truth  : ['_deplete_battery']


AttributeError: 'SMCDELGenerator' object has no attribute '_expr_cache'

In [ ]:

if __name__ == "__main__":
    if len(sys.argv) != 3:
        print("Usage: python loader.py <input.yaml> <output.txt>")
        sys.exit(1)
    yaml_to_smcdel(sys.argv[1], sys.argv[2])


In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Generate an SMCDEL file from a scenario ID."
    )
    parser.add_argument("yaml", help="Path to system_description.yaml")
    parser.add_argument("--scenario", type=int, required=True,
                        help="Scenario ID to run (e.g. 8)")
    parser.add_argument("--output", type=str, default=None,
                        help="Optional output file path (e.g. out.smcdel.txt)")
    args = parser.parse_args()

    run_scenario(
        yaml_path=args.yaml,
        scenario_id=args.scenario,
        output_path=args.output,
    )